# Converting TheWell GSDR data to PDEBench Format

Given that the data structures of both are very similar, the conversion is relatively straight forward.

1) Transpose the data order to match PDEBench DR's convention

Observation: DPOT does not seem to be able to generalize well to TheWell's GSDR data.

## NOTE
I have already converted the following data to PDEBench DR format for another model (MORPH) that I was working on before this, so check under MORPH's folder for conversion scripts.

In [33]:
#Have to adjust the data timesteps from 1000 to 100, as DPOT takes only 100 timesteps for dr_pdb

import h5py

input_path = '/Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles.h5'
output_file = '/Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles_last100t.h5'

PDB_min_A = -5.374724
PDB_max_A = 5.586630

GSDR_min = -0.002177
GSDR_max = 1.001419

def pde_normalize(data, pdb_min, pdb_max, gs_min, gs_max):
    """
    Maps data from ~[0, 1] (Gray-Scott) to [pdb_min, pdb_max] (PDEBench)
    Formula: x_new = (x_old - gsdr_min)/(gsdr_max - gsdr_min) * (pdb_max - pdb_min) + pdb_min
    """
    return (data - gs_min)/(gs_max - gs_min) * (pdb_max - pdb_min) + pdb_min

with h5py.File(input_path, 'r') as f_in, h5py.File(output_file, 'w') as f_out:
    for sim in f_in.keys():
        sim_in = f_in[sim]
        sim_out = f_out.create_group(sim)

        data_downsampled = sim_in['data'][-100:, :, :, :]

        data_normalized = pde_normalize(data_downsampled, PDB_min, PDB_max, GSDR_min, GSDR_max)

        sim_out.create_dataset('data', data=data_normalized, dtype='f4')

        grid_in = sim_in['grid']
        grid_out = sim_out.create_group('grid')

        t_downsampled = grid_in['t'][-100:]
        grid_out.create_dataset('t', data=t_downsampled, dtype='f4')

        grid_out.create_dataset('x', data=grid_in['x'][:], dtype='f4')
        grid_out.create_dataset('y', data=grid_in['y'][:], dtype='f4')

        print(f"Done {sim}: {data_downsampled.shape}")

print("All done")


Done 0000: (100, 128, 128, 2)
Done 0001: (100, 128, 128, 2)
Done 0002: (100, 128, 128, 2)
Done 0003: (100, 128, 128, 2)
Done 0004: (100, 128, 128, 2)
Done 0005: (100, 128, 128, 2)
Done 0006: (100, 128, 128, 2)
Done 0007: (100, 128, 128, 2)
Done 0008: (100, 128, 128, 2)
Done 0009: (100, 128, 128, 2)
Done 0010: (100, 128, 128, 2)
Done 0011: (100, 128, 128, 2)
Done 0012: (100, 128, 128, 2)
Done 0013: (100, 128, 128, 2)
Done 0014: (100, 128, 128, 2)
Done 0015: (100, 128, 128, 2)
Done 0016: (100, 128, 128, 2)
Done 0017: (100, 128, 128, 2)
Done 0018: (100, 128, 128, 2)
Done 0019: (100, 128, 128, 2)
All done


In [34]:
with h5py.File(output_file, 'r') as f_out:
    print(f_out.keys())
    print(f_out['0000']['data'])
    print(f_out['0000']['grid'])
    print(f_out['0000']['grid']['t'])

<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019']>
<HDF5 dataset "data": shape (100, 128, 128, 2), type "<f4">
<HDF5 group "/0000/grid" (3 members)>
<HDF5 dataset "t": shape (100,), type "<f4">


In [35]:
from DPOT.data_generation.preprocess import process_dr_pdebench

process_dr_pdebench(
    path = '/Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles_last100t.h5',
    save_name = '/Volumes/T7/TheWell/grayscott_PDB/DPOT_Processed',
    n_train = 0,
    n_test = 20
)

path created
(20, 128, 128, 100, 2)
train ids []
test ids [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
task @ 0 saved, shape (128, 128, 100, 2)
task @ 1 saved, shape (128, 128, 100, 2)
task @ 2 saved, shape (128, 128, 100, 2)
task @ 3 saved, shape (128, 128, 100, 2)
task @ 4 saved, shape (128, 128, 100, 2)
task @ 5 saved, shape (128, 128, 100, 2)
task @ 6 saved, shape (128, 128, 100, 2)
task @ 7 saved, shape (128, 128, 100, 2)
task @ 8 saved, shape (128, 128, 100, 2)
task @ 9 saved, shape (128, 128, 100, 2)
task @ 10 saved, shape (128, 128, 100, 2)
task @ 11 saved, shape (128, 128, 100, 2)
task @ 12 saved, shape (128, 128, 100, 2)
task @ 13 saved, shape (128, 128, 100, 2)
task @ 14 saved, shape (128, 128, 100, 2)
task @ 15 saved, shape (128, 128, 100, 2)
task @ 16 saved, shape (128, 128, 100, 2)
task @ 17 saved, shape (128, 128, 100, 2)
task @ 18 saved, shape (128, 128, 100, 2)
task @ 19 saved, shape (128, 128, 100, 2)
file saved


In [36]:
!python DPOT/evaluate.py \
    --model DPOT \
    --dataset dr_pdb \
    --train_paths dr_pdb \
    --test_paths dr_pdb \
    --resume_path DPOT/model_S.pth \
    --n_channels 4 \
    --T_in 10 \
    --res 128 \
    --batch_size 1 \
    --ntest_list 20 \
    --gpu cpu

Current working directory: /Users/sky/Git/DPOT
args Namespace(model='DPOT', dataset='dr_pdb', train_paths=['dr_pdb'], test_paths=['dr_pdb'], resume_path='DPOT/model_S.pth', ntrain_list=[100], ntest_list=[20], data_weights=[1], use_writer=False, res=128, noise_scale=0.0, width=1024, n_layers=6, act='gelu', max_nodes=-1, modes=20, use_ln=0, normalize=0, patch_size=8, n_blocks=8, mlp_ratio=1, out_layer_dim=32, batch_size=1, epochs=500, lr=0.001, opt='adam', beta1=0.9, beta2=0.999, lr_method='step', grad_clip=10000.0, step_size=100, step_gamma=0.5, warmup_epochs=50, sub=1, T_in=10, T_ar=1, T_bundle=1, gpu='cpu', comment='', log_path='', n_channels=4, n_class=12)
Train num [100] test num [20]
Loading models and fine tune from DPOT/model_S.pth
Using step learning rate schedule
DPOTNet(
  (pos_embed): tensor((1, 1024, 16, 16), requires_grad=True)
  
  (act): GELU(approximate='none')
  (patch_embed): PatchEmbed(
    (act): GELU(approximate='none')
    (proj): Sequential(
      (0): Conv2d(7, 3

In [38]:
import h5py
import numpy as np
from tqdm import tqdm

file_path = '/Volumes/T7/PDEBench/2D/diffusion-reaction/2D_diff-react_NA_NA.h5'

# Initialize separate trackers
min_a, max_a = float('inf'), float('-inf')
min_b, max_b = float('inf'), float('-inf')

with h5py.File(file_path, 'r') as f:
    sim_keys = list(f.keys())
    print(f"Analyzing {len(sim_keys)} simulations for per-channel range...")

    for key in tqdm(sim_keys):
        # data shape is  (t, x, y, c)
        data = f[key]['data'][()]

        # Split channels
        data_a = data[..., 0]
        data_b = data[..., 1]

        # Update A
        min_a = min(min_a, np.min(data_a))
        max_a = max(max_a, np.max(data_a))

        # Update B
        min_b = min(min_b, np.min(data_b))
        max_b = max(max_b, np.max(data_b))

    x_pdb = f['0000']['grid']['x'][:]
    print(f"PDEBench X range: [{x_pdb.min()}, {x_pdb.max()}]")

    y_pdb = f['0000']['grid']['y'][:]
    print(f"PDEBench X range: [{y_pdb.min()}, {y_pdb.max()}]")

print("\n--- PER-CHANNEL SANITY CHECK ---")
print(f"File: {file_path}")
print(f"Species A | Min: {min_a:.6f} | Max: {max_a:.6f}")
print(f"Species B | Min: {min_b:.6f} | Max: {max_b:.6f}")

# Check for global range as well
print(f"\nOverall Global Range: [{min(min_a, min_b):.6f}, {max(max_a, max_b):.6f}]")

Analyzing 1000 simulations for per-channel range...


100%|██████████| 1000/1000 [00:14<00:00, 66.70it/s]

PDEBench X range: [-0.994140625, 0.990234375]
PDEBench X range: [-0.994140625, 0.990234375]

--- PER-CHANNEL SANITY CHECK ---
File: /Volumes/T7/PDEBench/2D/diffusion-reaction/2D_diff-react_NA_NA.h5
Species A | Min: -5.279711 | Max: 5.586630
Species B | Min: -5.374724 | Max: 5.144024

Overall Global Range: [-5.374724, 5.586630]


In [39]:
import h5py
import numpy as np
from tqdm import tqdm

file_path = '/Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles.h5'

# Initialize separate trackers
min_a, max_a = float('inf'), float('-inf')
min_b, max_b = float('inf'), float('-inf')

with h5py.File(file_path, 'r') as f:
    sim_keys = list(f.keys())
    print(f"Analyzing {len(sim_keys)} simulations for per-channel range...")

    for key in tqdm(sim_keys):
        # data shape is typically (Time, X, Y, Channels)
        data = f[key]['data'][()]

        # Split channels
        data_a = data[..., 0]
        data_b = data[..., 1]

        # Update A
        min_a = min(min_a, np.min(data_a))
        max_a = max(max_a, np.max(data_a))

        # Update B
        min_b = min(min_b, np.min(data_b))
        max_b = max(max_b, np.max(data_b))

    x_pdb = f['0000']['grid']['x'][:]
    print(f"PDEBench X range: [{x_pdb.min()}, {x_pdb.max()}]")

    y_pdb = f['0000']['grid']['y'][:]
    print(f"PDEBench X range: [{y_pdb.min()}, {y_pdb.max()}]")


print("\n--- PER-CHANNEL SANITY CHECK ---")
print(f"File: {file_path}")
print(f"Species A | Min: {min_a:.6f} | Max: {max_a:.6f}")
print(f"Species B | Min: {min_b:.6f} | Max: {max_b:.6f}")

# Check for global range as well
print(f"\nOverall Global Range: [{min(min_a, min_b):.6f}, {max(max_a, max_b):.6f}]")

Analyzing 20 simulations for per-channel range...


100%|██████████| 20/20 [00:02<00:00,  6.79it/s]

PDEBench X range: [-1.0, 1.0]
PDEBench X range: [-1.0, 1.0]

--- PER-CHANNEL SANITY CHECK ---
File: /Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles.h5
Species A | Min: 0.000702 | Max: 1.001419
Species B | Min: -0.002177 | Max: 0.999523

Overall Global Range: [-0.002177, 1.001419]


In [24]:
import h5py
import numpy as np
from tqdm import tqdm

file_path = '/Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles_100t.h5'

# Initialize separate trackers
min_a, max_a = float('inf'), float('-inf')
min_b, max_b = float('inf'), float('-inf')

with h5py.File(file_path, 'r') as f:
    sim_keys = list(f.keys())
    print(f"Analyzing {len(sim_keys)} simulations for per-channel range...")

    for key in tqdm(sim_keys):
        # data shape is (t, x, y, c)
        data = f[key]['data'][()]

        # Split channels
        data_a = data[..., 0]
        data_b = data[..., 1]

        # Update A
        min_a = min(min_a, np.min(data_a))
        max_a = max(max_a, np.max(data_a))

        # Update B
        min_b = min(min_b, np.min(data_b))
        max_b = max(max_b, np.max(data_b))

print("\n--- PER-CHANNEL SANITY CHECK ---")
print(f"File: {file_path}")
print(f"Species A | Min: {min_a:.6f} | Max: {max_a:.6f}")
print(f"Species B | Min: {min_b:.6f} | Max: {max_b:.6f}")

# Check for global range as well
print(f"\nOverall Global Range: [{min(min_a, min_b):.6f}, {max(max_a, max_b):.6f}]")

Analyzing 20 simulations for per-channel range...


100%|██████████| 20/20 [00:00<00:00, 69.45it/s]


--- PER-CHANNEL SANITY CHECK ---
File: /Volumes/T7/TheWell/grayscott_PDB/gsdr_bubbles_100t.h5
Species A | Min: -5.343274 | Max: 5.578104
Species B | Min: -5.372953 | Max: 5.565919

Overall Global Range: [-5.372953, 5.578104]


In [2]:
# TheWell GSDR -> PDB DR
from DPOT.data_generation.preprocess import process_dr_pdebench

process_dr_pdebench(
    path = '/Volumes/T7/MORPH_Processed/DR2d_data_pdebench/gsdr_converted_pdbformat.h5',
    save_name = '/Volumes/T7/TheWell/grayscott_PDB/DPOT_Processed',
    n_train = 0,
    n_test = 20
)

path created
(20, 128, 128, 1001, 2)
train ids []
test ids [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
task @ 0 saved, shape (128, 128, 1001, 2)
task @ 1 saved, shape (128, 128, 1001, 2)
task @ 2 saved, shape (128, 128, 1001, 2)
task @ 3 saved, shape (128, 128, 1001, 2)
task @ 4 saved, shape (128, 128, 1001, 2)
task @ 5 saved, shape (128, 128, 1001, 2)
task @ 6 saved, shape (128, 128, 1001, 2)
task @ 7 saved, shape (128, 128, 1001, 2)
task @ 8 saved, shape (128, 128, 1001, 2)
task @ 9 saved, shape (128, 128, 1001, 2)
task @ 10 saved, shape (128, 128, 1001, 2)
task @ 11 saved, shape (128, 128, 1001, 2)
task @ 12 saved, shape (128, 128, 1001, 2)
task @ 13 saved, shape (128, 128, 1001, 2)
task @ 14 saved, shape (128, 128, 1001, 2)
task @ 15 saved, shape (128, 128, 1001, 2)
task @ 16 saved, shape (128, 128, 1001, 2)
task @ 17 saved, shape (128, 128, 1001, 2)
task @ 18 saved, shape (128, 128, 1001, 2)
task @ 19 saved, shape (128, 128, 1001, 2)
file saved


In [3]:
!python DPOT/evaluate.py \
    --model DPOT \
    --dataset dr_pdb \
    --train_paths dr_pdb \
    --test_paths dr_pdb \
    --resume_path DPOT/model_S.pth \
    --n_channels 4 \
    --T_in 10 \
    --res 128 \
    --batch_size 1 \
    --ntest_list 20 \
    --gpu cpu

Current working directory: /Users/sky/Git/DPOT
args Namespace(model='DPOT', dataset='dr_pdb', train_paths=['dr_pdb'], test_paths=['dr_pdb'], resume_path='DPOT/model_S.pth', ntrain_list=[100], ntest_list=[20], data_weights=[1], use_writer=False, res=128, noise_scale=0.0, width=1024, n_layers=6, act='gelu', max_nodes=-1, modes=20, use_ln=0, normalize=0, patch_size=8, n_blocks=8, mlp_ratio=1, out_layer_dim=32, batch_size=1, epochs=500, lr=0.001, opt='adam', beta1=0.9, beta2=0.999, lr_method='step', grad_clip=10000.0, step_size=100, step_gamma=0.5, warmup_epochs=50, sub=1, T_in=10, T_ar=1, T_bundle=1, gpu='cpu', comment='', log_path='', n_channels=4, n_class=12)
Train num [100] test num [20]
Loading models and fine tune from DPOT/model_S.pth
Using step learning rate schedule
DPOTNet(
  (pos_embed): tensor((1, 1024, 16, 16), requires_grad=True)
  
  (act): GELU(approximate='none')
  (patch_embed): PatchEmbed(
    (act): GELU(approximate='none')
    (proj): Sequential(
      (0): Conv2d(7, 3

In [4]:
import h5py

file = '/Volumes/T7/PDEBench/2D/diffusion-reaction/2D_diff-react_NA_NA.h5'

with h5py.File(file, 'r') as f:
    # print(f.keys())
    print(f['0000']['data'])

file2 = '/Volumes/T7/MORPH_Processed/DR2d_data_pdebench/TheWell_GSDR_converted/gsdr_converted_pdbformat.h5'

with h5py.File(file2, 'r') as f:
    print(f.keys())
    print(f['0000']['data'])

file3 = '/Volumes/T7/MORPH_Processed/DR2d_data_pdebench/gs_alpha_converted_pdbdr.h5'
with h5py.File(file3, 'r') as f:
    print(f.keys())
    print(f['0000']['data'])

<HDF5 dataset "data": shape (101, 128, 128, 2), type "<f4">
<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019']>
<HDF5 dataset "data": shape (1001, 128, 128, 2), type "<f4">
<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020', '0021', '0022', '0023', '0024', '0025', '0026', '0027', '0028', '0029']>
<HDF5 dataset "data": shape (100, 128, 128, 2), type "<f4">


# Converting APEBench GSDR data to PDEBench Format

Similar to TheWell GSDR, fairly straightforward to convert as well.

Taking only 'sims' from APEBench's GSDR:

1) Transpose the data order to match PDEBench DR's convention
2) Downsample the data from 256x256 to 128x128

Observation: DPOT does not seem to be able to generalize well to APEBench's GSDR data.

In [7]:
# APEBench GSDR -> PDB DR
from DPOT.data_generation.preprocess import process_dr_pdebench

process_dr_pdebench(
    path = '/Volumes/T7/MORPH_Processed/DR2d_data_pdebench/gs_alpha_converted_pdbdr.h5',
    save_name = '/Volumes/T7/APEBench/DPOT_Processed',
    n_train = 0,
    n_test = 30
)

path created
(30, 128, 128, 100, 2)
train ids []
test ids [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
task @ 0 saved, shape (128, 128, 100, 2)
task @ 1 saved, shape (128, 128, 100, 2)
task @ 2 saved, shape (128, 128, 100, 2)
task @ 3 saved, shape (128, 128, 100, 2)
task @ 4 saved, shape (128, 128, 100, 2)
task @ 5 saved, shape (128, 128, 100, 2)
task @ 6 saved, shape (128, 128, 100, 2)
task @ 7 saved, shape (128, 128, 100, 2)
task @ 8 saved, shape (128, 128, 100, 2)
task @ 9 saved, shape (128, 128, 100, 2)
task @ 10 saved, shape (128, 128, 100, 2)
task @ 11 saved, shape (128, 128, 100, 2)
task @ 12 saved, shape (128, 128, 100, 2)
task @ 13 saved, shape (128, 128, 100, 2)
task @ 14 saved, shape (128, 128, 100, 2)
task @ 15 saved, shape (128, 128, 100, 2)
task @ 16 saved, shape (128, 128, 100, 2)
task @ 17 saved, shape (128, 128, 100, 2)
task @ 18 saved, shape (128, 128, 100, 2)
task @ 19 saved, shape (128, 128, 100, 2)
task @ 20 saved, sh

In [8]:
!python DPOT/evaluate.py \
    --model DPOT \
    --dataset dr_pdb \
    --train_paths dr_pdb \
    --test_paths dr_pdb \
    --resume_path DPOT/model_S.pth \
    --n_channels 4 \
    --T_in 10 \
    --res 128 \
    --batch_size 1 \
    --ntest_list 30 \
    --gpu cpu

Current working directory: /Users/sky/Git/DPOT
args Namespace(model='DPOT', dataset='dr_pdb', train_paths=['dr_pdb'], test_paths=['dr_pdb'], resume_path='DPOT/model_S.pth', ntrain_list=[100], ntest_list=[30], data_weights=[1], use_writer=False, res=128, noise_scale=0.0, width=1024, n_layers=6, act='gelu', max_nodes=-1, modes=20, use_ln=0, normalize=0, patch_size=8, n_blocks=8, mlp_ratio=1, out_layer_dim=32, batch_size=1, epochs=500, lr=0.001, opt='adam', beta1=0.9, beta2=0.999, lr_method='step', grad_clip=10000.0, step_size=100, step_gamma=0.5, warmup_epochs=50, sub=1, T_in=10, T_ar=1, T_bundle=1, gpu='cpu', comment='', log_path='', n_channels=4, n_class=12)
Train num [100] test num [30]
Loading models and fine tune from DPOT/model_S.pth
Using step learning rate schedule
DPOTNet(
  (pos_embed): tensor((1, 1024, 16, 16), requires_grad=True)
  
  (act): GELU(approximate='none')
  (patch_embed): PatchEmbed(
    (act): GELU(approximate='none')
    (proj): Sequential(
      (0): Conv2d(7, 3